# ☁️ Classical Approach for Aerial Cloud Segmentation

This notebook presents the **classical machine learning** part of our cloud masking project for satellite images.

We selected **Random Forest** for this task based on its simplicity, interpretability, and strong performance on structured visual features, as demonstrated in the paper:

> Mapoka, S. (2022). *A Random Forest and Edge Vector Ensemble Model for Segmenting Aerial Satellite Forest Images*.  
> [Read the paper](https://www.oajaiml.com/uploads/archivepdf/403444180.pdf)

### 🔍 Why Random Forest?

- Works well with low- and mid-level image features (color, texture, edge vectors)
- Robust to noise and avoids overfitting
- No GPU required — efficient for quick experimentation


> 🔧 **Note:** This notebook covers only the classical part of the project. Deep learning experiments are documented separately.

In [ ]:
!pip install tifffile

## 📦 Imports, Setup, and Data Paths

We start by importing all required libraries for data handling, image processing, machine learning, and visualization.  
To ensure reproducibility, we set a fixed random seed for all relevant libraries.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import tifffile
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import random
import os
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
import time

# Set seed for reproducibility
SEED = 0
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define paths (adjust to your dataset location)
DATASET_DIR = Path("/kaggle/input/cloud-masking-dataset/content/train")
IMGS_DIR = DATASET_DIR / "data"
MASKS_DIR = DATASET_DIR / "masks"
IMG_SIZE = 512

# Get image and mask paths
image_paths = sorted(list(IMGS_DIR.glob("*.tif")))
mask_paths = sorted(list(MASKS_DIR.glob("*.tif")))

print(f"Total images found: {len(image_paths)}")

## 🏷️ Per-Pixel Feature Extraction for Random Forest

We extract per-pixel features and labels from each image and mask, sampling a fixed number of pixels per image for efficiency.  
This prepares the data in a format suitable for training the Random Forest classifier.

In [ ]:
from torch.utils.data import Dataset
import torch
import tifffile
import numpy as np

def read_tif_image(file_path):
    """Read a TIFF image and return a normalized PyTorch tensor on CPU."""
    img = tifffile.imread(file_path).astype(np.float32)  # [H, W, 4]
    img = torch.from_numpy(img).permute(2, 0, 1)  # [4, H, W]
    img = img / 255.0  # Normalize to [0, 1]
    return img 

def read_tif_mask(file_path):
    """Read a TIFF mask and return a PyTorch tensor on CPU."""
    mask = tifffile.imread(file_path).astype(np.float32)  # [H, W] or [H, W, 1]
    if mask.ndim == 2:
        mask = mask[..., None]  # Add channel dimension
    mask = torch.from_numpy(mask).permute(2, 0, 1)  # [1, H, W]
    return mask  # Keep on CPU

def parse_image_mask(img_path, mask_path):
    """Load, resize, and preprocess image and mask on CPU."""
    image = read_tif_image(img_path)  # [4, H, W]
    mask = read_tif_mask(mask_path)   # [1, H, W]
    
    IMG_SIZE = 512
    image = torch.nn.functional.interpolate(image.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).squeeze(0)
    mask = torch.nn.functional.interpolate(mask.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode='nearest').squeeze(0)
    
    mask = (mask > 0.5).float()
    
    return image, mask

def augmentation(image, mask):
    """Apply random horizontal and vertical flips."""
    import random
    if random.random() > 0.5:
        image = torch.flip(image, dims=[2])  # Horizontal flip
        mask = torch.flip(mask, dims=[2])
    if random.random() > 0.5:
        image = torch.flip(image, dims=[1])  # Vertical flip
        mask = torch.flip(mask, dims=[1])
    return image, mask

class CloudDataset(Dataset):
    """Custom Dataset for cloud masking images and masks."""
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment
    
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img_path = str(self.img_paths[idx])
        mask_path = str(self.mask_paths[idx])
        image, mask = parse_image_mask(img_path, mask_path)
        if self.augment:
            image, mask = augmentation(image, mask)
        return image, mask

## 🏗️ Dataset Preparation and Splitting

We split our dataset into training, validation, and test sets using a fixed random seed for reproducibility.  
For each image, we extract a fixed number of randomly sampled pixels and their corresponding mask values to use as features and labels for the Random Forest.

- **prepare_dataset:**  
  Samples per-pixel features from each image and mask, returning arrays ready for classical machine learning.
- **Splitting:**  
  - 100 images are reserved for the test set.
  - 20% of the remaining images are used for validation.
  - The rest are used for training.

This approach ensures efficient training and evaluation while keeping the test set completely unseen until final assessment.

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
import random

def prepare_dataset(img_paths, mask_paths, batch_size=8, device='cuda', pixels_per_image=1000):
    """
    Prepare per-pixel features and labels for Random Forest, sampling pixels per image.
    Returns X (features), y (labels) as NumPy arrays.
    """
    dataset = CloudDataset(img_paths, mask_paths, augment=False)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    
    # Lists to collect features and labels
    features = []
    labels = []
    
    total_batches = len(dataloader)
    np.random.seed(0)  # Ensure reproducible pixel sampling
    
    for i, (images, masks) in enumerate(dataloader):
        # I/O and GPU transfer
        try:
            images, masks = images.to(device), masks.to(device)
        except RuntimeError as e:
            print(f"Error moving batch {i} to {device}: {e}")
            continue
        
        # Convert to NumPy and extract per-pixel features
        images_np = images.cpu().numpy()  # [B, 4, H, W]
        masks_np = masks.cpu().numpy()    # [B, 1, H, W]
        
        # Sample pixels per image
        for j in range(images_np.shape[0]):
            img = images_np[j].transpose(1, 2, 0)  # [512, 512, 4]
            mask = masks_np[j].squeeze(0)          # [512, 512]
            img_flat = img.reshape(-1, 4)          # [512*512, 4]
            mask_flat = mask.ravel()               # [512*512]
            # Randomly sample pixels
            indices = np.random.choice(512*512, size=pixels_per_image, replace=False)
            features.append(img_flat[indices])      # [pixels_per_image, 4]
            labels.append(mask_flat[indices])       # [pixels_per_image]
        
        # Free memory
        del images, masks, images_np, masks_np
        torch.cuda.empty_cache()
        
        # Print progress every 10 batches
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{total_batches} batches")
    
    # Convert lists to NumPy arrays
    X = np.vstack(features)  # Shape: [n_samples*pixels_per_image, 4]
    y = np.hstack(labels)    # Shape: [n_samples*pixels_per_image]
    
    return X, y

# Split data into train, validation, and test sets with fixed seed
random.seed(0)  # Ensure reproducible split
combined = list(zip(image_paths, mask_paths))
random.shuffle(combined)

# Test set: 100 images
test_size = 100
test_set = combined[:test_size]
remaining = combined[test_size:]

# Validation set: 20% of remaining
val_size = int(0.2 * len(remaining))
val_set = remaining[:val_size]
train_set = remaining[val_size:]

# Unzip paths
train_imgs, train_masks = zip(*train_set)
val_imgs, val_masks = zip(*val_set)
test_imgs, test_masks = zip(*test_set)

train_imgs = list(train_imgs)
train_masks = list(train_masks)
val_imgs = list(val_imgs)
val_masks = list(val_masks)
test_imgs = list(test_imgs)
test_masks = list(test_masks)

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

# Clear GPU memory
torch.cuda.empty_cache()

# Prepare datasets
start_time = time.time()
X_train, y_train = prepare_dataset(train_imgs, train_masks, batch_size=8, pixels_per_image=1000)
X_val, y_val = prepare_dataset(val_imgs, val_masks, batch_size=8, pixels_per_image=1000)
end_time = time.time()

print(f"Training samples: {X_train.shape[0]}, Validation samples: {X_val.shape[0]}")
print(f"Training feature shape: {X_train.shape}, Label shape: {y_train.shape}")
print(f"Time passed: {end_time - start_time:.2f} seconds")

## 🏁 Final Training on All Available Data

After selecting the best hyperparameters, we retrain the Random Forest model using both the training and validation data combined.  
This ensures the model benefits from all labeled data before final evaluation on the unseen test set.

- Features are standardized using `StandardScaler`.
- The model is trained with the optimal settings found during validation.
- Training time is reported for reference.

The trained model will be evaluated on the test set using the Dice coefficient in the next step.

In [ ]:
import time
import psutil
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Initialize StandardScaler
scaler = StandardScaler()

# Combine train and validation sets
X_full = np.vstack([X_train, X_val])
y_full = np.hstack([y_train, y_val])

# Standardize features
start_time = time.time()
try:
    X_full_scaled = scaler.fit_transform(X_full)
except Exception as e:
    print(f"Error during scaling: {e}")
    raise

# Initialize Random Forest
rf_model = RandomForestClassifier(
    n_estimators=30,      # 30 (10, 30, 50, 100)
    max_depth=20,         # (10,20, 30, None)
    min_samples_split=5,  # 5 (2,5,10)
    min_samples_leaf=2,   # 2 (1,2,5)
    max_features='sqrt',  # 'sqrt'& 'log2' are almost same and much faster than None
    class_weight=None,    # None wins over 'balanced'
    criterion='gini',     # 'gini' wins over 'entropy' by small difference
    random_state=SEED,
    n_jobs=-1
)

try:
    print("Starting Training on full data (train + val)")
    rf_model.fit(X_full_scaled, y_full)
except Exception as e:
    print(f"Error during training: {e}")
    raise
end_time = time.time()

print(f"Training time: {end_time - start_time:.2f} seconds")

## 🧪 Test Set Evaluation

We evaluate the trained model on the test set using the Dice coefficient, which measures segmentation quality.  
Dice scores are computed in parallel for all test images for faster evaluation.

In [ ]:
from joblib import Parallel, delayed

def predict_mask(model, image, scaler, img_size=512):
    image = F.interpolate(image.unsqueeze(0), size=(img_size, img_size), mode='bilinear', align_corners=False).squeeze(0)
    features = image.cpu().numpy().transpose(1, 2, 0).reshape(-1, 4)  # [512*512, 4]
    features = scaler.transform(features)
    pred_labels = model.predict(features)
    pred_mask = pred_labels.reshape(img_size, img_size)
    return pred_mask.astype(np.uint8)

def dice_coefficient(mask1, mask2):
    """Computes Dice coefficient between two binary masks."""
    intersection = np.sum(mask1 * mask2)
    return (2.0 * intersection) / (np.sum(mask1) + np.sum(mask2) + 1e-7)

def compute_dice_for_image(img_path, mask_path):
    image, true_mask = parse_image_mask(img_path, mask_path)
    pred_mask = predict_mask(rf_model, image, scaler)
    dice = dice_coefficient(true_mask.cpu().numpy().squeeze(), pred_mask)
    return dice

start_time = time.time()

# Parallel Dice computation for test set
raw_dice_scores = Parallel(n_jobs=-1, prefer="threads")(
    delayed(compute_dice_for_image)(img_path, mask_path)
    for img_path, mask_path in zip(test_imgs, test_masks)
)

end_time = time.time()

print(f"Mean Dice on test set: {np.mean(raw_dice_scores):.4f}")
print(f"Test set evaluation time: {end_time - start_time:.2f} seconds")

## 🖼️ Visualizing Predictions

We randomly sample a few test images to visualize the input, ground truth mask, predicted mask, and an overlay of the prediction.  
This helps qualitatively assess the segmentation performance of the Random Forest model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

def normalize_img(img):
    """Normalize RGB for visualization."""
    img = img.cpu().numpy().transpose(1, 2, 0)[:, :, :3]
    out = np.zeros_like(img, dtype=np.float32)
    for i in range(3):
        band = img[:, :, i]
        min_val = np.percentile(band, 2)
        max_val = np.percentile(band, 98)
        out[:, :, i] = np.clip((band - min_val) / (max_val - min_val + 1e-6), 0, 1)
    return out

def display_predictions(image_list, mask_list, model, scaler, n_images=3):
    """Display n_images randomly sampled images, masks, and overlays."""
    # Randomly sample n_images indices
    indices = random.sample(range(len(image_list)), min(n_images, len(image_list)))
    
    # Create figure with n_images rows and 4 columns
    plt.figure(figsize=(16, 4 * n_images))
    
    for i, idx in enumerate(indices):
        # Load image and mask
        img_path, mask_path = image_list[idx], mask_list[idx]
        image, true_mask = parse_image_mask(img_path, mask_path)
        pred_mask = predict_mask(model, image, scaler)
        
        # Normalize image and prepare masks
        rgb_img = normalize_img(image)
        true_mask = true_mask.cpu().numpy().squeeze()
        pred_mask = pred_mask.squeeze()
        
        # Plot Input Image
        plt.subplot(n_images, 4, i * 4 + 1)
        plt.imshow(rgb_img)
        plt.title("Input Image")
        plt.axis("off")
        
        # Plot True Mask
        plt.subplot(n_images, 4, i * 4 + 2)
        plt.imshow(true_mask, cmap='gray')
        plt.title("True Mask")
        plt.axis("off")
        
        # Plot Predicted Mask
        plt.subplot(n_images, 4, i * 4 + 3)
        plt.imshow(pred_mask, cmap='gray')
        plt.title("Predicted Mask")
        plt.axis("off")
        
        # Plot Overlay
        plt.subplot(n_images, 4, i * 4 + 4)
        plt.imshow(rgb_img)
        plt.imshow(pred_mask, alpha=0.4, cmap='jet')
        plt.title("Overlay: Prediction")
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

# Visualize n_images randomly sampled validation images
display_predictions(test_imgs, test_masks, rf_model, scaler, n_images=3)